In [19]:
# import pandas as pd

# # RICHTIGER Datensatz:
# df = pd.read_csv("Data_Files/base_data104_with_lookup.csv", low_memory=False)

# df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
# df = df.dropna(subset=["Country", "Year"])

# # Mapping
# country_map = {
#     "United Kingdom, England and Wales": "United Kingdom",
#     "United Kingdom, Scotland": "United Kingdom",
#     "United Kingdom, Northern Ireland": "United Kingdom",
#     "United States of America": "United States",
# }

# df["Country_clean"] = df["Country"].replace(country_map)


# counts = (
#     df.groupby(["Country_clean", "Year"])
#       .size()
#       .reset_index(name="Entries")
# )

# all_years = sorted(counts["Year"].unique())
# end_year = max(all_years)
# start_year = end_year - 20

# years_to_plot = list(range(start_year, end_year + 1, 2))
# years_to_plot

# all_countries = sorted(df["Country_clean"].unique())

# full = pd.MultiIndex.from_product(
#     [all_countries, years_to_plot],
#     names=["Country_clean", "Year"]
# )

# counts_full = (
#     counts.set_index(["Country_clean", "Year"])
#           .reindex(full, fill_value=0)
#           .reset_index()
# )


# import plotly.express as px
# import plotly.io as pio

# # Browser als Renderer
# pio.renderers.default = "browser"

# fig = px.choropleth(
#     counts_full,
#     locations="ISO3",                    # <-- ISO-3 statt Ländermame
#     color="Entries",
#     animation_frame="Year",
#     color_continuous_scale="Viridis",
#     title="WHO Mortality – Datenverfügbarkeit pro Land (20 Jahre, 2-Jahres-Schritte)",
#     range_color=(0, counts_full["Entries"].max()),
# )

# fig.write_html("WHO_Data_Animation.html")
# fig.show()


import pandas as pd
import pycountry
import plotly.express as px
import plotly.io as pio
import imageio
import os

# -------------------------------------------------------
# 1) Load WHO data
# -------------------------------------------------------
df = pd.read_csv("Data_Files/base_data104_with_lookup.csv", low_memory=False)
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df = df.dropna(subset=["Country", "Year"])

# clean names
country_map = {
    "United Kingdom, England and Wales": "United Kingdom",
    "United Kingdom, Scotland": "United Kingdom",
    "United Kingdom, Northern Ireland": "United Kingdom",
    "United States of America": "United States",
}
df["Country_clean"] = df["Country"].replace(country_map)

# ISO
def to_iso3(name):
    try:
        return pycountry.countries.lookup(name).alpha_3
    except:
        return None

df["ISO3"] = df["Country_clean"].apply(to_iso3)

# entries per year
counts = (
    df.groupby(["ISO3", "Year"])
      .size()
      .reset_index(name="Entries")
)

# pick 13 years (2010–2022)
years = list(range(2010, 2023))

# ensure all countries appear each year
all_iso = counts["ISO3"].dropna().unique()
full_index = pd.MultiIndex.from_product([all_iso, years], names=["ISO3", "Year"])

counts_full = (
    counts.set_index(["ISO3", "Year"])
          .reindex(full_index, fill_value=0)
          .reset_index()
)

# Zero = None → country not colored
counts_full.loc[counts_full["Entries"] == 0, "Entries"] = None

# folder for frames
os.makedirs("gif_frames", exist_ok=True)

# -------------------------------------------------------
# 2) CREATE 13 PNG FILES
# -------------------------------------------------------
for year in years:
    sub = counts_full[counts_full["Year"] == year]

    fig = px.choropleth(
        sub,
        locations="ISO3",
        color="Entries",
        color_continuous_scale="Viridis",
        title=f"WHO Mortality – Data Availability {year}",
        range_color=(0, counts_full["Entries"].max())
    )

    fig.update_layout(width=1100, height=700)

    fname = f"gif_frames/frame_{year}.png"
    pio.write_image(fig, fname)
    print("Saved:", fname)

# -------------------------------------------------------
# 3) BUILD GIF
# -------------------------------------------------------
images = []
for year in years:
    img = imageio.imread(f"gif_frames/frame_{year}.png")
    images.append(img)

# duration controls speed between frames (0.7s = chill)
imageio.mimsave("WHO_Data_13_Years3.gif", images, duration=100)

print("✔ DONE — GIF saved as WHO_Data_13_Years.gif")



Saved: gif_frames/frame_2010.png
Saved: gif_frames/frame_2011.png
Saved: gif_frames/frame_2012.png
Saved: gif_frames/frame_2013.png
Saved: gif_frames/frame_2014.png
Saved: gif_frames/frame_2015.png
Saved: gif_frames/frame_2016.png
Saved: gif_frames/frame_2017.png
Saved: gif_frames/frame_2018.png
Saved: gif_frames/frame_2019.png
Saved: gif_frames/frame_2020.png
Saved: gif_frames/frame_2021.png
Saved: gif_frames/frame_2022.png


/var/folders/37/084722ds0s5f0lsp5b75z_w40000gn/T/ipykernel_38758/1609368507.py:151: DeprecationWarning:

Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.



✔ DONE — GIF saved as WHO_Data_13_Years.gif
